# Pricing across asset classes

**Prerequisites:** notebook **05** in this curriculum (the standard pricing pipeline: tagged instrument JSON, `MarketContext` as JSON, valuation date, model key, and `ValuationResult.from_json`). This notebook reuses that same pipeline while swapping the **instrument** and, when needed, the **model** and **metrics**.

**Goal:** see one consistent workflow for rates, credit, equity, and FX—without leaving the JSON-oriented API.

## Concepts: a quick tour of asset classes

Finstack groups instruments by **risk factor** and **cashflow shape**, not by desk org chart. The same pricing entry points apply everywhere:

- **Rates (money market & swaps):** PV from discounting (and forward projection for float legs). Curves: OIS discount, IBOR/RFR **forward** where needed.
- **Credit (CDS):** protection vs premium legs; survival from **hazard** curves. Model key is typically `hazard_rate` rather than pure `discounting`.
- **Equity options:** Black–Scholes–style models (`black76`) with **spot**, **discount**, **dividend yield**, and an **implied vol surface**.
- **FX options:** Garman–Kohlhagen-style stack: **two discount curves** (domestic/foreign), **FX spot** in the matrix, and a **vol surface** keyed by expiry × strike.

The pattern in each section below is deliberate and repeatable:

1. Build tagged instrument JSON (`{"type": "…", "spec": {…}}`).
2. Build a **fresh** `MarketContext`, serialize with `to_json()`, and (for FX) **patch** `fx.quotes`—see the setup cell for why.
3. Call `price_instrument` or `price_instrument_with_metrics` with the right **model** string.
4. Parse with `ValuationResult.from_json` and read `price`, `currency`, and optional `get_metric`.

In [ ]:
import json
from datetime import date

from finstack.core.currency import Currency
from finstack.core.market_data import (
    DiscountCurve,
    ForwardCurve,
    FxMatrix,
    HazardCurve,
    MarketContext,
)
from finstack.valuations import (
    ValuationResult,
    list_standard_metrics,
    price_instrument,
    price_instrument_with_metrics,
    validate_instrument_json,
)

metrics_sample = sorted(list_standard_metrics())
print("total standard metrics:", len(metrics_sample))
print("sample:", metrics_sample[:18])

## Shared market data (self-contained)

We construct curves on **`as_of = 2025-01-15`**: USD OIS discount, USD SOFR 3M forward, a single-name hazard curve for CDS, and EUR OIS for FX option domestic/foreign discounting.

**FX caveat:** `FxMatrix.set_quote` updates the live matrix, but the current Python snapshot serializer can emit **`fx.quotes: []`**. For **FX options** we patch quotes in the decoded JSON (`[["EUR", "USD", rate]]`) before re-serializing. Equity and rates legs above ignore FX.

We append **vol surfaces** and **equity spot / dividend yield** scalars to the decoded state so option pricers can resolve their IDs.

In [ ]:
AS_OF_DATE = date(2025, 1, 15)
AS_OF = AS_OF_DATE.isoformat()


def build_market_context_json(as_of: date) -> str:
    """Build a JSON market snapshot suitable for `price_instrument*`."""
    ois_usd = DiscountCurve(
        "USD-OIS",
        as_of,
        [(0.0, 1.0), (0.5, 0.985), (1.0, 0.97), (3.0, 0.90), (5.0, 0.82), (10.0, 0.65)],
        day_count="act_365f",
    )
    ois_eur = DiscountCurve(
        "EUR-OIS",
        as_of,
        [(0.0, 1.0), (0.5, 0.988), (1.0, 0.975), (3.0, 0.92), (5.0, 0.85), (10.0, 0.70)],
        day_count="act_365f",
    )
    sofr_3m = ForwardCurve(
        "USD-SOFR-3M",
        0.25,
        [(0.0, 0.052), (1.0, 0.048), (3.0, 0.045), (5.0, 0.043), (10.0, 0.041)],
        base_date=as_of,
        day_count="act_360",
    )
    hazard = HazardCurve(
        "CORP-HAZARD",
        as_of,
        [(0.5, 0.018), (1.0, 0.020), (3.0, 0.024), (5.0, 0.028), (10.0, 0.032)],
        recovery_rate=0.40,
    )
    fx = FxMatrix()
    fx.set_quote(Currency("EUR"), Currency("USD"), 1.10)

    mc = MarketContext()
    mc.insert(ois_usd).insert(ois_eur).insert(sofr_3m).insert(hazard).insert_fx(fx)

    state = json.loads(mc.to_json())
    # Ensure EUR/USD is visible to JSON-only pricers (see markdown above).
    state["fx"]["quotes"] = [["EUR", "USD", 1.10]]

    exp_eq = [0.25, 0.5, 1.0, 2.0]
    strikes_eq = [4000.0, 4300.0, 4500.0, 4700.0, 5000.0]
    exp_fx = [0.25, 0.5, 1.0, 2.0]
    strikes_fx = [0.9, 1.0, 1.1, 1.2, 1.3]
    state["surfaces"] = [
        {
            "id": "EQUITY-VOL",
            "expiries": exp_eq,
            "strikes": strikes_eq,
            "secondary_axis": "strike",
            "interpolation_mode": "vol",
            "vols_row_major": [0.22] * (len(exp_eq) * len(strikes_eq)),
        },
        {
            "id": "EURUSD-VOL",
            "expiries": exp_fx,
            "strikes": strikes_fx,
            "secondary_axis": "strike",
            "interpolation_mode": "vol",
            "vols_row_major": [0.12] * (len(exp_fx) * len(strikes_fx)),
        },
    ]
    state.setdefault("prices", {})
    state["prices"]["EQUITY-SPOT"] = {"price": {"amount": 4500.0, "currency": "USD"}}
    state["prices"]["EQUITY-DIVYIELD"] = {"unitless": 0.015}

    return json.dumps(state)


MARKET_JSON = build_market_context_json(AS_OF_DATE)
ROWS: list[dict] = []

print("as_of:", AS_OF)
print("market JSON length (chars):", len(MARKET_JSON))
print("surfaces:", len(json.loads(MARKET_JSON)["surfaces"]))

## 1. Deposit (rates / money market)

A **deposit** is the simplest discounting exercise: fixed coupon on a known day-count between `start_date` and `maturity`. Model: **`discounting`**.

We optionally run `validate_instrument_json` to pretty-print the canonical wire format.

In [ ]:
deposit = json.dumps(
    {
        "type": "deposit",
        "spec": {
            "id": "DEP-1",
            "notional": {"amount": 1_000_000.0, "currency": "USD"},
            "start_date": "2025-01-15",
            "maturity": "2025-06-15",
            "day_count": "Act360",
            "quote_rate": 0.05,
            "discount_curve_id": "USD-OIS",
            "attributes": {},
        },
    }
)

canon = validate_instrument_json(deposit)
print("canonical instrument JSON (first 220 chars):\n", canon[:220], "…")

out = price_instrument_with_metrics(
    deposit, MARKET_JSON, AS_OF, model="discounting", metrics=["dv01"]
)
vr = ValuationResult.from_json(out)
print(vr)
print("dv01:", vr.get_metric("dv01"))

ROWS.append(
    {
        "class": "Rates",
        "instrument_id": vr.instrument_id,
        "price": vr.price,
        "ccy": vr.currency,
        "key_metric": "dv01",
        "metric_value": vr.get_metric("dv01"),
    }
)

## 2. Interest rate swap (IRS)

Plain **fixed-for-floating** swap: fixed leg vs SOFR-linked float with `forward_curve_id`. Still **`discounting`** for the standard registry path.

**Seasoning:** if the floating leg has resets **before** `as_of`, the engine may require historical fixings. Here we use an **unseasoned** swap starting `2025-04-15`.

In [ ]:
irs = json.dumps(
    {
        "type": "interest_rate_swap",
        "spec": {
            "id": "IRS-5Y-USD",
            "notional": {"amount": 10_000_000.0, "currency": "USD"},
            "side": "pay",
            "fixed": {
                "discount_curve_id": "USD-OIS",
                "rate": 0.03,
                "frequency": {"count": 6, "unit": "months"},
                "day_count": "Thirty360",
                "bdc": "modified_following",
                "calendar_id": None,
                "stub": "None",
                "start": "2025-04-15",
                "end": "2030-04-15",
                "par_method": None,
                "compounding_simple": True,
            },
            "float": {
                "discount_curve_id": "USD-OIS",
                "forward_curve_id": "USD-SOFR-3M",
                "spread_bp": 0.0,
                "frequency": {"count": 3, "unit": "months"},
                "day_count": "Act360",
                "bdc": "modified_following",
                "calendar_id": None,
                "stub": "None",
                "reset_lag_days": 2,
                "start": "2025-04-15",
                "end": "2030-04-15",
                "compounding": "Simple",
            },
            "attributes": {},
        },
    }
)

try:
    out = price_instrument_with_metrics(
        irs, MARKET_JSON, AS_OF, model="discounting", metrics=["dv01"]
    )
    vr = ValuationResult.from_json(out)
    print(vr)
    print("dv01:", vr.get_metric("dv01"))
    ROWS.append(
        {
            "class": "Rates",
            "instrument_id": vr.instrument_id,
            "price": vr.price,
            "ccy": vr.currency,
            "key_metric": "dv01",
            "metric_value": vr.get_metric("dv01"),
        }
    )
except Exception as exc:
    print("IRS pricing skipped:", type(exc).__name__, "—", exc)
    ROWS.append(
        {
            "class": "Rates",
            "instrument_id": "IRS-5Y-USD",
            "price": None,
            "ccy": "",
            "key_metric": "dv01",
            "metric_value": None,
        }
    )

## 3. Credit default swap (CDS)

Single-name **CDS**: premium leg vs protection leg tied to `credit_curve_id` (`CORP-HAZARD` here). Use model **`hazard_rate`**.

JSON **`convention`** uses snake_case (e.g. `isda_na`), not `IsdaNa`.

In [ ]:
cds = json.dumps(
    {
        "type": "credit_default_swap",
        "spec": {
            "id": "CDS-CORP-5Y",
            "notional": {"amount": 10_000_000.0, "currency": "USD"},
            "side": "pay",
            "convention": "isda_na",
            "premium": {
                "start": "2025-03-20",
                "end": "2030-03-20",
                "frequency": {"count": 3, "unit": "months"},
                "stub": "ShortFront",
                "bdc": "following",
                "calendar_id": None,
                "day_count": "Act360",
                "spread_bp": 100.0,
                "discount_curve_id": "USD-OIS",
            },
            "protection": {
                "credit_curve_id": "CORP-HAZARD",
                "recovery_rate": 0.4,
                "settlement_delay": 3,
            },
            "pricing_overrides": {},
            "attributes": {},
        },
    }
)

try:
    out = price_instrument_with_metrics(
        cds, MARKET_JSON, AS_OF, model="hazard_rate", metrics=["cs01"]
    )
    vr = ValuationResult.from_json(out)
    print(vr)
    print("cs01:", vr.get_metric("cs01"))
    ROWS.append(
        {
            "class": "Credit",
            "instrument_id": vr.instrument_id,
            "price": vr.price,
            "ccy": vr.currency,
            "key_metric": "cs01",
            "metric_value": vr.get_metric("cs01"),
        }
    )
except Exception as exc:
    print("CDS pricing skipped:", type(exc).__name__, "—", exc)
    ROWS.append(
        {
            "class": "Credit",
            "instrument_id": "CDS-CORP-5Y",
            "price": None,
            "ccy": "",
            "key_metric": "cs01",
            "metric_value": None,
        }
    )

## 4. Equity option

Vanilla **equity** call: `black76` model, `spot_id` / `vol_surface_id` / optional `div_yield_id` resolved from the market snapshot (`EQUITY-SPOT`, `EQUITY-VOL`, `EQUITY-DIVYIELD`).

The wire schema uses a **numeric** `strike` and `notional` as `Money` (not `contract_size`).

In [ ]:
equity_option = json.dumps(
    {
        "type": "equity_option",
        "spec": {
            "id": "SPX-CALL-4500",
            "underlying_ticker": "SPX",
            "strike": 4500.0,
            "option_type": "call",
            "exercise_style": "european",
            "expiry": "2026-06-15",
            "notional": {"amount": 100.0, "currency": "USD"},
            "day_count": "Act365F",
            "settlement": "cash",
            "discount_curve_id": "USD-OIS",
            "spot_id": "EQUITY-SPOT",
            "vol_surface_id": "EQUITY-VOL",
            "div_yield_id": "EQUITY-DIVYIELD",
            "pricing_overrides": {},
            "attributes": {},
        },
    }
)

try:
    out = price_instrument_with_metrics(
        equity_option,
        MARKET_JSON,
        AS_OF,
        model="black76",
        metrics=["delta"],
    )
    vr = ValuationResult.from_json(out)
    print(vr)
    print("delta:", vr.get_metric("delta"))
    ROWS.append(
        {
            "class": "Equity",
            "instrument_id": vr.instrument_id,
            "price": vr.price,
            "ccy": vr.currency,
            "key_metric": "delta",
            "metric_value": vr.get_metric("delta"),
        }
    )
except Exception as exc:
    print("Equity option pricing skipped:", type(exc).__name__, "—", exc)
    ROWS.append(
        {
            "class": "Equity",
            "instrument_id": "SPX-CALL-4500",
            "price": None,
            "ccy": "",
            "key_metric": "delta",
            "metric_value": None,
        }
    )

## 5. FX option

EUR/USD **call**: domestic discount `USD-OIS`, foreign `EUR-OIS`, `vol_surface_id` `EURUSD-VOL`, and FX spot from the (patched) matrix.

Model: **`black76`**. We request **`delta`** here (registered for this pricer); it is expressed in the option’s quoting currency and scales with notional.

In [ ]:
fx_option = json.dumps(
    {
        "type": "fx_option",
        "spec": {
            "id": "FXOPT-EURUSD-CALL",
            "base_currency": "EUR",
            "quote_currency": "USD",
            "strike": 1.12,
            "option_type": "call",
            "exercise_style": "european",
            "expiry": "2026-06-15",
            "day_count": "Act365F",
            "notional": {"amount": 1_000_000.0, "currency": "EUR"},
            "settlement": "cash",
            "domestic_discount_curve_id": "USD-OIS",
            "foreign_discount_curve_id": "EUR-OIS",
            "vol_surface_id": "EURUSD-VOL",
            "pricing_overrides": {},
            "attributes": {},
        },
    }
)

try:
    out = price_instrument_with_metrics(
        fx_option,
        MARKET_JSON,
        AS_OF,
        model="black76",
        metrics=["delta"],
    )
    vr = ValuationResult.from_json(out)
    print(vr)
    print("delta:", vr.get_metric("delta"))
    ROWS.append(
        {
            "class": "FX",
            "instrument_id": vr.instrument_id,
            "price": vr.price,
            "ccy": vr.currency,
            "key_metric": "delta",
            "metric_value": vr.get_metric("delta"),
        }
    )
except Exception as exc:
    print("FX option pricing skipped:", type(exc).__name__, "—", exc)
    ROWS.append(
        {
            "class": "FX",
            "instrument_id": "FXOPT-EURUSD-CALL",
            "price": None,
            "ccy": "",
            "key_metric": "delta",
            "metric_value": None,
        }
    )

## Mini-example: side-by-side results

The table below collects **instrument id**, **PV**, **currency**, and one **requested metric** per row. Metrics are not uniform across asset classes (rates `dv01`, credit `cs01`, options `delta`)—that is intentional: it mirrors how desks ask different risk questions.

In [ ]:
hdr = f"{'class':<8} {'instrument_id':<16} {'price':>14} {'ccy':<4} {'metric':<10} {'value':>12}"
print(hdr)
print("-" * len(hdr))
for row in ROWS:
    p = row["price"]
    p_str = f"{p:,.4f}" if p is not None else "n/a"
    m = row["metric_value"]
    m_str = f"{m:,.6f}" if m is not None else "n/a"
    print(
        f"{row['class']:<8} {row['instrument_id']:<16} {p_str:>14} "
        f"{row['ccy']:<4} {row['key_metric']:<10} {m_str:>12}"
    )
print("rows:", len(ROWS))

## Takeaways

- **One pipeline:** `validate_instrument_json` → `price_instrument*` → `ValuationResult.from_json` works across rates, credit, equity, and FX; notebook **05** is the reference for that skeleton.
- **Model keys matter:** `discounting` for deposits and swaps, `hazard_rate` for CDS, `black76` for vanilla equity and FX options in the default registry wiring.
- **Market JSON is explicit:** curves must cover every `*_curve_id`; options need **surfaces** and **scalars** (`prices`); FX options also need a **non-empty** `fx.quotes` snapshot when using `to_json()` from Python—patch if quotes serialize empty.
- **Deep dives:** follow the curriculum notebooks on **interest_rate_swap** / **credit_default_swap** / **equity_option** / **fx_option** instrument JSON (and the foundations notebook on **curves and `MarketContext`**) for fuller schedules, conventions, and metric sets.

**Cross-references (repo):** instrument schemas under `finstack/valuations/schemas/instruments/`, worked JSON under `finstack/valuations/tests/instruments/json_examples/`, and the market snapshot format in `finstack/core` (`MarketContext` serde / `VolSurface` grid layout: `vols_row_major`).